# 9 WorkFlow Analista Jr

## Proyecto Problema #02 - DR Data Drifting (Grupo A) - version 6.2

Este notebook corre **un metodo de drifting por vez** (`PARAM$DR$metodo`) y evalua con **10 semillas**.

Flujo:

1. Preprocesamiento (Catastrophe Analysis, Data Drifting, features historicos).
2. **Grid Search 3x3** con la semilla primigenia (165463): `num_leaves` x `min_data_in_leaf`.
3. **Bucle de semillas**: para cada una de las 10 semillas
   * modelo de validacion (solo training) -> probabilidades en 202107 (se **guardan**), AUC y ganancia local,
   * modelo final (final_train) -> probabilidades para 202109 (se guardan).
4. **Ensamble**: promedio de las probabilidades de las 10 semillas.
5. Envio a Kaggle del ensamble en **cortes fijos e iguales para todos los metodos** (1600 a 2600 cada 100).

Novedades de la version 6.2 (respecto de la version 6.1):

* **Semillerio de primos (opcional)**: `PARAM$semillerio$modo`. Con `"fijas"` (por defecto) se usan las 10 semillas del grupo, igual que en todos los experimentos (los resultados de las versiones 6 y 6.1 no cambian). Con `"primos"` se eligen `PARAM$semillerio$cantidad` primos al azar entre 100.000 y 1.000.000, con la semilla primigenia como semilla del sorteo (el semillerio de la catedra). Pensado para la corrida final con mas semillas. Los notebooks de Wilcoxon y CV asumen las semillas fijas.

Novedades de la version 6.1 (respecto de la version 6):

* **Metodo nuevo `deflacion_selectiva`** (carpeta `WF9609`, al final de `metodos_DR` para no mover los numeros de carpeta anteriores): deflacta con el IPC todas las variables monetarias **salvo** las de `PARAM$DR$no_deflactar` (`mrentabilidad`, `mrentabilidad_annual`, `mpasivos_margen`, `mprestamos_personales`), que quedan en pesos corrientes. El resto del workflow es identico.
* Criterio de la lista: variables que casi no crecieron en pesos (`crec_vs_IPC` < 0.6) y a las que la deflacion les agrega tendencia. Es provisoria y se puede editar.

Novedades de la version 6 (respecto de la version 5):

* **Semillas 6 a 10 nuevas**: 100183, 250027, 400031, 550049, 700057 (reemplazan a 264137, 374137, 529229, 627269, 724783). Las semillas 1 a 5 no cambian. Las corridas de la version 5 (`WF94xx`) usaron las semillas viejas.
* **UVA sale de `metodos_DR`** (no esta en el diseno del experimento). La funcion `drift_UVA` sigue definida pero el metodo ya no se puede elegir.
* **Ponderacion temporal**: dos metodos nuevos, `ninguno_pond` y `deflacion_pond` (el drifting `ninguno` / `deflacion` mas pesos por registro en el entrenamiento). Peso = 0.5 ^ (meses de distancia al ultimo mes de entrenamiento / `PARAM$pond$vida_media_meses`), normalizado a promedio 1. Los pesos se aplican en el Grid Search, en el modelo de validacion y en el modelo final; la validacion no se pondera.
* **Variables externas opcionales** (riesgo pais, tasas internacionales): `PARAM$externos$archivo` apunta a un CSV con `foto_mes` mas las columnas nuevas. Vacio = no se usa.
* **Experimento base 9600** (`WF9600` ... `WF9608`). El orden de `metodos_DR` define el numero de carpeta.
* El **test de Wilcoxon** esta en un notebook aparte (`z620_Wilcoxon_DR_grupoA`), que lee las carpetas de todos los metodos.

Novedades de la version 5 (respecto de la version 4):

* **Una carpeta distinta por corrida**: el experimento pasa a 9400 (`WF9400`, `WF9401`, ...) y el notebook **se detiene** si la carpeta ya existe y tiene archivos (`PARAM$permitir_carpeta_existente`). Para una corrida nueva se sube `PARAM$experimento_base` (9400 -> 9500 -> ...).

Novedades de la version 4 (respecto de la version 3):

* Se guardan las **probabilidades de validation por semilla** (`prediccion_val_<semilla>.txt`) y la clase real (`validacion_clases.txt`): con eso se puede recalcular la ganancia con cualquier costo, o cualquier otra metrica, SIN reentrenar.
* **10 semillas**: se agregan las semillas 6 a 10 (primos entre 190003 y 999983, elegidos con `set.seed(165463)`).
* **Reanudacion** (`PARAM$reanudar`): lo que ya esta guardado en la carpeta del experimento se lee del disco en lugar de recalcularse. Asi se pueden sumar semillas a un experimento ya corrido.

Cambios de la version 3 respecto del test 3x3 (experimento 9200):

* Experimentos 9xxx (cada corrida en su carpeta): `WF9200` queda intacto como referencia de una semilla.
* Grilla nueva: `min_data_in_leaf` {2048, 4096, 8192} (el mejor valor, 2048, estaba en el borde de la grilla anterior) y `num_leaves` {128, 256, 512}.
* Los cortes de Kaggle ya **no** se eligen por ganancia local: en el test 3x3 la ganancia local ubico el optimo en 2550 y Kaggle en 2200 o menos. Al ser fijos, todos los metodos se comparan en los mismos cortes.
* Se mantienen las correcciones del test 3x3 (switch de drifting, `mpayroll_sobre_edad` antes del drifting, `modifyList`, `foto_mes` y `numero_de_cliente` fuera del modelo, NA en `rank_simple`, `Corregir_MICE`, `Corregir_interpolar`).

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Nov 06 22:32:22 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671183,35.9,1479644,79.1,1479644,79.1
Vcells,1242599,9.5,8388608,64.0,1978693,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros

In [7]:
PARAM <- list()

# semillas del proyecto: la primera es la primigenia de TODOS los experimentos
#  semillas 1 a 5: definidas por el grupo
#  semillas 6 a 10: definidas el 2026-10-31 (reemplazan a las de la version 5)
#    (las de la version 5 eran 264137, 374137, 529229, 627269, 724783: NO usar)
PARAM$semillas <- c(165463, 329969, 454253, 593899, 980249,
                    100183, 250027, 400031, 550049, 700057)
PARAM$semilla_primigenia <- PARAM$semillas[1]

# semillas que se corren en el bucle (para una prueba rapida: PARAM$semillas[1])
PARAM$semillas_a_correr <- PARAM$semillas

# TRUE = lo que ya existe en la carpeta del experimento (prediccion_<semilla>.txt y
#  prediccion_val_<semilla>.txt) se lee del disco y NO se reentrena.
#  En una carpeta nueva no hay nada que leer; solo sirve para retomar una corrida cortada
#  (junto con PARAM$permitir_carpeta_existente <- TRUE).
PARAM$reanudar <- TRUE

# metodo de Data Drifting que se corre en este experimento
#  *_pond = mismo drifting que el metodo base + ponderacion temporal en el entrenamiento
#  OJO: el ORDEN de esta lista define el numero de carpeta (y es el mismo del notebook de Wilcoxon)
metodos_DR <- c(
  "ninguno", "deflacion", "dolar_blue", "dolar_oficial",
  "rank_simple", "rank_cero_fijo", "estandarizar",
  "ninguno_pond", "deflacion_pond",
  "deflacion_selectiva"   # agregado AL FINAL para no cambiar los numeros de carpeta (WF9609)
)
PARAM$DR$metodo <- "ninguno"

if( !(PARAM$DR$metodo %in% metodos_DR) )
  stop( paste0("metodo de drifting desconocido: ", PARAM$DR$metodo) )

# metodo de drifting propiamente dicho, y si se pondera o no el entrenamiento
PARAM$DR$ponderar <- grepl( "_pond$", PARAM$DR$metodo )
PARAM$DR$metodo_base <- sub( "_pond$", "", PARAM$DR$metodo )

# deflacion_selectiva: variables que se dejan en pesos corrientes (el resto se deflacta con el IPC)
#  criterio (diagnostico del drifting, WF9600): variables que casi no crecieron en pesos
#  (crec_vs_IPC < 0.6) y a las que la deflacion les agrega tendencia (|rho| sube mas de 0.15)
PARAM$DR$no_deflactar <- c(
  "mrentabilidad", "mrentabilidad_annual", "mpasivos_margen", "mprestamos_personales"
)

# CADA corrida nueva va en su propia carpeta WF<experimento>. Al empezar una corrida nueva
#  se sube PARAM$experimento_base a la centena siguiente (9400 -> 9500 -> ...).
#  Historial: 9200 = test 3x3 (1 semilla), 9300 = primera corrida con 5 semillas, 9400 = 10 semillas (6 a 10 viejas), 9500 = notebook de 10 semillas en una sola carpeta,
#   9600 = semillas 6 a 10 nuevas + ponderacion temporal
PARAM$experimento_base <- 10100

# 9600 = ninguno, 9601 = deflacion, 9602 = dolar_blue, ... (segun el orden de metodos_DR)
PARAM$experimento <- PARAM$experimento_base + match(PARAM$DR$metodo, metodos_DR) - 1

# FALSE = si la carpeta del experimento ya existe y tiene archivos, el notebook se detiene
#  (evita mezclar o pisar corridas). TRUE solo para reanudar una corrida que se corto.
PARAM$permitir_carpeta_existente <- FALSE
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

# ponderacion temporal (solo metodos *_pond):
#  peso = 0.5 ^ (meses de distancia al ultimo mes de entrenamiento / vida_media), normalizado a promedio 1
PARAM$pond$vida_media_meses <- 12

# variables externas opcionales (riesgo pais, tasas internacionales, ...)
#  CSV con la columna foto_mes + las columnas nuevas, UNA fila por foto_mes.
#  Los nombres de las columnas nuevas NO pueden empezar con m, Visa_m, Master_m ni vm_m
#  (el notebook las trataria como variables monetarias). "" = no se usa.
PARAM$externos$archivo <- ""

# campos que NO entran al modelo (vaciar el vector para volver al comportamiento original)
PARAM$features$excluir <- c("foto_mes", "numero_de_cliente")

# funcion de ganancia: solo BAJA+2 suma
#  SE USA SOLO para la ganancia local de referencia en validation.
#  Los cortes de Kaggle son fijos, asi que NO dependen de estos valores.
PARAM$ganancia$acierto <- 975000
PARAM$ganancia$costo <- -25000   # SUPUESTO, confirmar con la pestaña Evaluation de Kaggle

# Kaggle: cortes FIJOS, iguales para todos los metodos
PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"
PARAM$kaggle$cortes <- seq(1600, 2600, by = 100)
PARAM$kaggle$enviar <- TRUE   # FALSE = genera los archivos pero NO los sube

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")

#### Carpeta del Experimento
Cada experimento se guarda en su propia carpeta `WF<experimento>`. Si la carpeta ya existe y tiene archivos, el notebook se detiene para no mezclar ni pisar corridas anteriores.

In [8]:
# carpeta de trabajo: CADA experimento va en una carpeta distinta (WF<experimento>)

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)

# si la carpeta ya existe y tiene archivos, NO se reutiliza (salvo que se pida expresamente)
if( dir.exists(experimento_folder) &&
    length( list.files(experimento_folder, all.files= TRUE, no.. = TRUE) ) > 0 &&
    !PARAM$permitir_carpeta_existente )
  stop( paste0( "La carpeta ", experimento_folder, " ya existe y no esta vacia.\n",
    "  Cada corrida nueva debe ir en una carpeta distinta: suba PARAM$experimento_base.\n",
    "  (para retomar una corrida cortada: PARAM$permitir_carpeta_existente <- TRUE)" ) )

dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

cat( "carpeta del experimento:", getwd(), "\n" )

carpeta del experimento: /home/ds/buckets/b1/exp/WF10100 


### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [9]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

# variables externas opcionales (se agregan por foto_mes, igual para todos los metodos)
if( nzchar(PARAM$externos$archivo) )
{
  externos <- fread(PARAM$externos$archivo)
  cols_ext <- setdiff( colnames(externos), "foto_mes" )

  stopifnot( "foto_mes" %in% colnames(externos), length(cols_ext) > 0,
             !anyDuplicated(externos$foto_mes),
             !any(cols_ext %in% colnames(dataset)) )
  if( any(cols_ext %like% "^(m|Visa_m|Master_m|vm_m)") )
    stop( "las columnas externas no pueden empezar con m, Visa_m, Master_m ni vm_m" )

  dataset[externos, on= "foto_mes", (cols_ext) := mget(paste0("i.", cols_ext))]
  cat( "columnas externas agregadas:", cols_ext, "\n" )
}

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [10]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [11]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  # corregido: se completa la imputacion, no el dataset
  tbl <- as.data.table( mice::complete(imputacion) )

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}

In [12]:
Corregir_interpolar <- function(pcampo, pmeses) {

  # corregido: la clave del cliente es numero_de_cliente (antes usaba envg, que no existe aca)
  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = numero_de_cliente
  ]

  tbl[, numero_de_cliente := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [13]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [14]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [15]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [16]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [17]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [18]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [19]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [20]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [21]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [22]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [23]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [24]:
drift_deflacion_selectiva <- function(campos_monetarios) {
  cat( "inicio drift_deflacion_selectiva()\n")

  # variables que NO se deflactan (quedan en pesos corrientes)
  fuera <- intersect( PARAM$DR$no_deflactar, campos_monetarios )
  falta <- setdiff( PARAM$DR$no_deflactar, campos_monetarios )
  if( length(falta) > 0 )
    cat( "ATENCION: no estan en el dataset y se ignoran:", falta, "\n" )

  campos <- setdiff( campos_monetarios, fuera )
  cat( "no se deflactan:", fuera, "\n" )
  cat( "se deflactan", length(campos), "de", length(campos_monetarios), "variables\n" )

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos) := .SD * i.IPC,
    .SDcols = campos
  ]

  cat( "fin drift_deflacion_selectiva()\n")
}


In [25]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    # corregido: na.last="keep" para que los NA (Catastrophe Analysis) sigan siendo NA;
    #  sin esto frank() les asigna un rank (en un mes todo NA, un rank al azar)
    #  el denominador cuenta solo los valores no NA del mes
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random", na.last = "keep") - 1) /
        (sum(!is.na(get(campo))) - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}

In [26]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [27]:
# funcion provista por alumnos amantes de la Estadistica Clasica

drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(get(campo), na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [28]:
# mpayroll_sobre_edad se crea ANTES del drifting, asi TODOS los metodos la reciben.
#  (con los rank, mpayroll desaparece y el feature se perdia en silencio)
#  como es un cociente de una variable monetaria, tambien se corrige por drifting
if( all( c("mpayroll", "cliente_edad") %in% colnames(dataset) ) )
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"                   "mpayroll_sobre_edad"

In [29]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )

# los rank con ties.method="random" consumen numeros aleatorios
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")

cat("metodo:", PARAM$DR$metodo, "  drifting:", PARAM$DR$metodo_base,
  "  ponderacion temporal:", PARAM$DR$ponderar, "\n")

switch(PARAM$DR$metodo_base,
  "ninguno"        = cat("No hay correccion del data drifting\n"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "deflacion_selectiva" = drift_deflacion_selectiva(campos_monetarios),
  "dolar_blue"     = drift_dolar_blue(campos_monetarios),
  "dolar_oficial"  = drift_dolar_oficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios),
  stop( paste0("metodo de drifting desconocido: ", PARAM$DR$metodo) )
)

metodo: ninguno   drifting: ninguno   ponderacion temporal: FALSE 
No hay correccion del data drifting


In [30]:
# control rapido: promedio mensual de un campo monetario (despues del drifting)
campo_control <- grep("^mcuentas_saldo", colnames(dataset), value= TRUE)[1]

if( !is.na(campo_control) )
  print( dataset[ foto_mes %in% c(201901, 201912, 202012, 202109),
    list( campo= campo_control, promedio= mean(get(campo_control), na.rm= TRUE), .N ),
    by= foto_mes ] )

   foto_mes          campo promedio     N
      <int>         <char>    <num> <int>
1:   201901 mcuentas_saldo 157597.8 24956
2:   201912 mcuentas_saldo 185419.2 27977
3:   202012 mcuentas_saldo 221042.4 32228
4:   202109 mcuentas_saldo 259144.3 33080


In [31]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "mpayroll_sobre_edad"

In [32]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [33]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# mpayroll_sobre_edad (variable extraida de una tesis de maestria de Irlanda)
#  se crea mas arriba, antes del Data Drifting

In [34]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "mpayroll_sobre_edad"               
[57] "kmes"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [35]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [36]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [37]:
ncol(dataset)
colnames(dataset)

[1] 273

[1] "numero_de_cliente"                        
  [2] "foto_mes"                                 
  [3] "internet"                                 
  [4] "cliente_edad"                             
  [5] "cliente_antiguedad"                       
  [6] "mrentabilidad"                            
  [7] "mrentabilidad_annual"                     
  [8] "mcomisiones"                              
  [9] "mactivos_margen"                          
 [10] "mpasivos_margen"                          
 [11] "cproductos"                               
 [12] "mcuenta_corriente"                        
 [13] "mcaja_ahorro"                             
 [14] "cdescubierto_preacordado"                 
 [15] "mcuentas_saldo"                           
 [16] "ctarjeta_visa"                            
 [17] "ctarjeta_visa_transacciones"              
 [18] "mtarjeta_visa_consumo"                    
 [19] "ctarjeta_master"                          
 [20] "ctarjeta_master_transacciones"            
 [21] "mtarjeta_master_consumo"                  
 [22] "cprestamos_personales"                    
 [23] "mprestamos_personales"                    
 [24] "cpayroll_trx"                             
 [25] "mpayroll"                                 
 [26] "mttarjeta_visa_debitos_automaticos"       
 [27] "ccomisiones_mantenimiento"                
 [28] "mcomisiones_mantenimiento"                
 [29] "ccomisiones_otras"                        
 [30] "mtransferencias_recibidas"                
 [31] "ccallcenter_transacciones"                
 [32] "thomebanking"                             
 [33] "chomebanking_transacciones"               
 [34] "ctrx_quarter"                             
 [35] "Master_status"                            
 [36] "Master_mfinanciacion_limite"              
 [37] "Master_Fvencimiento"                      
 [38] "Master_msaldototal"                       
 [39] "Master_mlimitecompra"                     
 [40] "Master_fultimo_cierre"                    
 [41] "Master_fechaalta"                         
 [42] "Master_mconsumototal"                     
 [43] "Master_cconsumos"                         
 [44] "Master_mpagominimo"                       
 [45] "Visa_status"                              
 [46] "Visa_mfinanciacion_limite"                
 [47] "Visa_Fvencimiento"                        
 [48] "Visa_msaldototal"                         
 [49] "Visa_mlimitecompra"                       
 [50] "Visa_fultimo_cierre"                      
 [51] "Visa_fechaalta"                           
 [52] "Visa_mconsumototal"                       
 [53] "Visa_cconsumos"                           
 [54] "Visa_mpagominimo"                         
 [55] "clase_ternaria"                           
 [56] "mpayroll_sobre_edad"                      
 [57] "kmes"                                     
 [58] "internet_lag1"                            
 [59] "cliente_edad_lag1"                        
 [60] "cliente_antiguedad_lag1"                  
 [61] "mrentabilidad_lag1"                       
 [62] "mrentabilidad_annual_lag1"                
 [63] "mcomisiones_lag1"                         
 [64] "mactivos_margen_lag1"                     
 [65] "mpasivos_margen_lag1"                     
 [66] "cproductos_lag1"                          
 [67] "mcuenta_corriente_lag1"                   
 [68] "mcaja_ahorro_lag1"                        
 [69] "cdescubierto_preacordado_lag1"            
 [70] "mcuentas_saldo_lag1"                      
 [71] "ctarjeta_visa_lag1"                       
 [72] "ctarjeta_visa_transacciones_lag1"         
 [73] "mtarjeta_visa_consumo_lag1"               
 [74] "ctarjeta_master_lag1"                     
 [75] "ctarjeta_master_transacciones_lag1"       
 [76] "mtarjeta_master_consumo_lag1"             
 [77] "cprestamos_personales_lag1"               
 [78] "mprestamos_personales_lag1"               
 [79] "cpayroll_trx_lag1"                        
 [80] "mpayroll_lag1"                            
 [

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [38]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [39]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [40]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [41]:
# los campos en los que se entrena
#  se excluyen la clase, el azar y los campos de PARAM$features$excluir
campos_buenos <- copy( setdiff(
    colnames(dataset),
    c("clase_ternaria", "clase01", "azar", PARAM$features$excluir))
)

##### Ponderacion temporal (solo metodos `*_pond`)
Pesos por registro segun la cercania del mes al ultimo mes de entrenamiento: peso = 0.5 ^ (distancia en meses / `PARAM$pond$vida_media_meses`), normalizado a promedio 1 para que la escala sea comparable con los metodos sin ponderar. En los metodos sin ponderacion la funcion devuelve `NULL` (sin pesos).

In [42]:
mes_indice <- function(foto_mes) (foto_mes %/% 100) * 12 + (foto_mes %% 100)

calc_pesos <- function(foto_mes, mes_ref)
{
  if( !PARAM$DR$ponderar ) return( NULL )

  w <- 0.5 ^ ( (mes_indice(mes_ref) - mes_indice(foto_mes)) / PARAM$pond$vida_media_meses )
  return( w / mean(w) )
}

In [43]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  weight= calc_pesos( dataset[fold_train == TRUE, foto_mes],
    max(PARAM$trainingstrategy$training) ),
  free_raw_data= TRUE
)

Loading required package: lightgbm



In [44]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search (3x3 = 9 corridas, con la semilla primigenia)
  * num_leaves  {128, 256, 512}
  * min_data_in_leaf  {2048, 4096, 8192}
  * feature_fraction queda fijo en 0.5

En el test 3x3 anterior el mejor `min_data_in_leaf` era 2048, justo el borde de la grilla; por eso ahora se exploran valores mas altos. La grilla es la **misma para todos los metodos de drifting**.

In [45]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [46]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [47]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(128, 256, 512),
  min_data_in_leaf= c(2048, 4096, 8192)
)

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> son 9 combinaciones (3x3); el Grid Search se corre UNA vez por metodo, con la semilla primigenia

In [48]:
# registro a registro calculo la AUC
tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Fri Nov 06 22:36:23 2026  128, 2048 niter 166 AUC 0.93240739859603

Fri Nov 06 22:37:41 2026  128, 4096 niter 253 AUC 0.932278664378528

Fri Nov 06 22:38:57 2026  128, 8192 niter 276 AUC 0.933316270101835

Fri Nov 06 22:40:13 2026  256, 2048 niter 176 AUC 0.934403043308623

Fri Nov 06 22:41:25 2026  256, 4096 niter 205 AUC 0.932057675044171

Fri Nov 06 22:42:43 2026  256, 8192 niter 276 AUC 0.933316270101835

Fri Nov 06 22:43:57 2026  512, 2048 niter 135 AUC 0.933212846564673

Fri Nov 06 22:45:11 2026  512, 4096 niter 205 AUC 0.932057675044171

Fri Nov 06 22:46:27 2026  512, 8192 niter 276 AUC 0.933316270101835



la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [49]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
128,2048,0.9324074,166
128,4096,0.9322787,253
128,8192,0.9333163,276
256,2048,0.9344030,176
256,4096,0.9320577,205
256,8192,0.9333163,276
512,2048,0.9332128,135
512,4096,0.9320577,205
512,8192,0.9333163,276


In [50]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 256

$min_data_in_leaf
[1] 2048

$num_iterations
[1] 176

In [51]:
tb_nueva

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
256,2048,0.9344030,176
128,8192,0.9333163,276
256,8192,0.9333163,276
512,8192,0.9333163,276
512,2048,0.9332128,135
128,2048,0.9324074,166
128,4096,0.9322787,253
256,4096,0.9320577,205
512,4096,0.9320577,205


### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [52]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)


dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  weight= calc_pesos( dataset[fold_final_train == TRUE, foto_mes],
    max(PARAM$trainingstrategy$final_train) ),
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 910853

##### Final Training Hyperparameters

In [53]:
# uno los parametros fijos y los mejores encontrados en el Grid Search
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en el Grid Search
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# modifyList PISA los valores fijos con los mejores encontrados
#  (con c() quedaban nombres duplicados: num_leaves y min_data_in_leaf aparecian dos veces)
param_final <- modifyList(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training por semilla
Para cada una de las semillas de `PARAM$semillas_a_correr` se entrenan dos modelos con los mejores hiperparametros del Grid Search:

* **modelo de validacion**: solo con `training`, evaluado en 202107 (AUC y ganancia local). Sirve para comparar metodos semilla por semilla. Sus probabilidades se **guardan** en `prediccion_val_<semilla>.txt`.
* **modelo final**: con todo `final_train` (sin undersampling), aplicado a 202109. Sus probabilidades se guardan en `prediccion_<semilla>.txt` y despues se promedian.

Archivos que quedan en la carpeta del experimento:

* `prediccion_val_<semilla>.txt`: `numero_de_cliente`, `prob` en 202107 (una por semilla).
* `validacion_clases.txt`: `numero_de_cliente`, `clase_ternaria` de 202107 (una sola vez). Junto con lo anterior permite recalcular AUC y ganancia con cualquier costo, fuera del notebook.
* `firma_hiperparametros.txt`: los mejores hiperparametros con los que se generaron los archivos por semilla.

**Reanudacion** (solo para retomar una corrida cortada, junto con `PARAM$permitir_carpeta_existente <- TRUE`): con `PARAM$reanudar <- TRUE`, si un archivo por semilla ya existe se lee en vez de recalcularse. Si los mejores hiperparametros cambiaron respecto de `firma_hiperparametros.txt`, el notebook se detiene (los archivos viejos ya no corresponden). Cada semilla se reanuda por separado en validation y en final.

##### Semillas del bucle (semillerio)

* `"fijas"` (por defecto): las 10 semillas del grupo (`PARAM$semillas`), las mismas que usaron todos los experimentos.
* `"primos"`: `PARAM$semillerio$cantidad` numeros primos al azar entre 100.000 y 1.000.000. El sorteo usa `PARAM$semilla_primigenia`, asi que es **reproducible**: la misma cantidad da siempre las mismas semillas. Poner la cantidad en 5 o en 1 si se esta muy ajustado de tiempo.

In [54]:
# modo del semillerio: "fijas" | "primos"
PARAM$semillerio$modo <- "fijas"

# cantidad de semillas en el modo "primos" (5 o 1 si se esta muy ajustado de tiempo)
PARAM$semillerio$cantidad <- 5

stopifnot( PARAM$semillerio$modo %in% c("fijas", "primos") )

if( PARAM$semillerio$modo == "primos" )
{
  # primos entre 100000 y 1000000: con el paquete primes si esta instalado; si no, una criba propia
  #  (da exactamente los mismos numeros)
  if( requireNamespace("primes", quietly= TRUE) ) {
    primos <- primes::generate_primes( min= 100000, max= 1000000 )
  } else {
    es_primo <- rep( TRUE, 1000000 )
    es_primo[1] <- FALSE
    for( i in 2:1000 ) if( es_primo[i] ) es_primo[ seq( i * i, 1000000, by= i ) ] <- FALSE
    primos <- which( es_primo )
    primos <- primos[ primos >= 100000 ]
  }

  # me quedo con PARAM$semillerio$cantidad primos al azar
  set.seed( PARAM$semilla_primigenia, kind= "L'Ecuyer-CMRG" )
  PARAM$semillas_a_correr <- sample( primos )[ seq( PARAM$semillerio$cantidad ) ]
}

cat( "modo:", PARAM$semillerio$modo, "| semillas del bucle:", PARAM$semillas_a_correr, "\n" )

modo: fijas | semillas del bucle: 165463 329969 454253 593899 980249 100183 250027 400031 550049 700057 


In [55]:
# --- datos de validation (202107) y del futuro (202109) en formato matriz ---------
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]
mfuture <- data.matrix( dfuture[, campos_buenos, with= FALSE] )

dval <- dataset[ foto_mes %in% PARAM$trainingstrategy$validate,
  list(numero_de_cliente, clase_ternaria, clase01) ]
mval <- data.matrix( dataset[ foto_mes %in% PARAM$trainingstrategy$validate,
  campos_buenos, with= FALSE] )

# --- metricas de validation -------------------------------------------------------

# AUC (por rangos) de la clase binaria que se optimiza en el Grid Search
AUC_rank <- function(score, label) {
  r <- frank(score, ties.method= "average")
  n1 <- sum(label == 1)
  n0 <- length(label) - n1
  ( sum(r[label == 1]) - n1 * (n1 + 1) / 2 ) / ( n1 * n0 )
}

# ganancia acumulada si se envian los primeros k de la lista ordenada por probabilidad
#  solo BAJA+2 suma  (ganancia de REFERENCIA: depende del costo supuesto)
ganancia_cortes <- function(prob, clase_ternaria, cortes) {
  orden <- order(prob, decreasing= TRUE)
  gan <- ifelse( clase_ternaria[orden] == "BAJA+2",
    PARAM$ganancia$acierto, PARAM$ganancia$costo )
  gan_acum <- cumsum(gan)
  gan_acum[ pmin( cortes, length(gan_acum) ) ]
}

# los cortes estan definidos sobre los clientes de 202109;
#  en validation se llevan a la misma proporcion de la poblacion
tb_cortes_val <- data.table( envios= PARAM$kaggle$cortes )
tb_cortes_val[, envios_val := pmax( 1L,
  as.integer( round( envios * nrow(dval) / nrow(dfuture) ) ) ) ]

In [56]:
# --- 1) preparo lo que se guarda una sola vez -------------------------------------

# clase real de validation: junto con prediccion_val_<semilla>.txt permite recalcular
#  AUC y ganancia (con cualquier costo) fuera del notebook, sin reentrenar
fwrite( dval[, list(numero_de_cliente, clase_ternaria)],
  file= "validacion_clases.txt", sep= "\t" )

# firma de los hiperparametros: si cambian, los archivos por semilla ya guardados
#  fueron generados con otros hiperparametros y no se pueden reutilizar
mh <- PARAM$out$lgbm$mejores_hiperparametros
firma_actual <- paste( names(mh), unlist(mh), sep= "=", collapse= ";" )
cat( "mejores hiperparametros:", firma_actual, "\n" )

if( PARAM$reanudar && file.exists("firma_hiperparametros.txt") ) {
  firma_previa <- readLines("firma_hiperparametros.txt")[1]
  if( firma_previa != firma_actual )
    stop( paste0( "los mejores hiperparametros cambiaron respecto de los archivos por semilla guardados.\n",
      "  antes: ", firma_previa, "\n  ahora: ", firma_actual, "\n",
      "  borre prediccion_*.txt o ponga PARAM$reanudar <- FALSE" ) )
}
writeLines( firma_actual, "firma_hiperparametros.txt" )

# lee las probabilidades de un archivo, alineadas con los clientes que corresponde
leer_prob <- function( archivo, clientes ) {
  tb <- fread( archivo )
  prob <- tb[ match( clientes, numero_de_cliente ), prob ]
  if( anyNA(prob) )
    stop( paste0( "el archivo ", archivo, " no coincide con los clientes actuales" ) )
  return( prob )
}

# --- 2) bucle de semillas ---------------------------------------------------------
lista_val <- list()    # probabilidades en validation, una por semilla
lista_prob <- list()   # probabilidades en 202109, una por semilla
lista_tb <- list()     # metricas de validation por semilla

for( semilla in PARAM$semillas_a_correr )
{
  s <- as.character( semilla )
  arch_val <- paste0( "prediccion_val_", s, ".txt" )
  arch_fut <- paste0( "prediccion_", s, ".txt" )

  cat( "semilla", s, " inicio", format(Sys.time(), "%X"), "\n" )

  # (a) validation: modelo con solo training, num_iterations = el del Grid Search
  if( PARAM$reanudar && file.exists( arch_val ) ) {

    cat( "  validation: se lee", arch_val, "\n" )
    prob_val <- leer_prob( arch_val, dval$numero_de_cliente )

  } else {

    param_val <- modifyList( PARAM$lgbm$param_fijos, mh )
    param_val$seed <- semilla
    param_val$early_stopping_rounds <- NULL

    modelo_val <- lgb.train( data= dtrain, param= param_val, verbose= -100 )
    prob_val <- predict( modelo_val, mval )

    # GUARDO las probabilidades de validation de esta semilla
    fwrite( data.table( numero_de_cliente= dval$numero_de_cliente, prob= prob_val ),
      file= arch_val, sep= "\t" )

    rm( modelo_val )
  }

  lista_val[[ s ]] <- prob_val
  lista_tb[[ s ]] <- data.table(
    semilla= s,
    auc_val= AUC_rank( prob_val, dval$clase01 ),
    envios= tb_cortes_val$envios,
    ganancia_val= ganancia_cortes( prob_val, dval$clase_ternaria, tb_cortes_val$envios_val )
  )

  # (b) final: modelo con todo final_train, sin undersampling
  if( PARAM$reanudar && file.exists( arch_fut ) ) {

    cat( "  final: se lee", arch_fut, "\n" )
    prob_fut <- leer_prob( arch_fut, dfuture$numero_de_cliente )

  } else {

    param_fin <- modifyList( param_final, list( seed= semilla ) )
    final_model <- lgb.train( data= dfinal_train, param= param_fin, verbose= -100 )

    lgb.save( final_model, paste0( "modelo_", s, ".txt" ) )
    fwrite( as.data.table( lgb.importance(final_model) ),
      file= paste0( "impo_", s, ".txt" ), sep= "\t" )

    prob_fut <- predict( final_model, mfuture )

    fwrite( data.table( numero_de_cliente= dfuture$numero_de_cliente, prob= prob_fut ),
      file= arch_fut, sep= "\t" )

    rm( final_model )
  }

  lista_prob[[ s ]] <- prob_fut

  gc( full= TRUE, verbose= FALSE )
  cat( "semilla", s, " fin", format(Sys.time(), "%X"), "\n" )
}

mejores hiperparametros: num_leaves=256;min_data_in_leaf=2048;num_iterations=176 
semilla 165463  inicio 22:46:30 
semilla 165463  fin 22:47:48 
semilla 329969  inicio 22:47:48 
semilla 329969  fin 22:49:04 
semilla 454253  inicio 22:49:04 
semilla 454253  fin 22:50:18 
semilla 593899  inicio 22:50:18 
semilla 593899  fin 22:51:31 
semilla 980249  inicio 22:51:31 
semilla 980249  fin 22:52:44 
semilla 100183  inicio 22:52:44 
semilla 100183  fin 22:53:57 
semilla 250027  inicio 22:53:57 
semilla 250027  fin 22:55:10 
semilla 400031  inicio 22:55:10 
semilla 400031  fin 22:56:25 
semilla 550049  inicio 22:56:25 
semilla 550049  fin 22:57:38 
semilla 700057  inicio 22:57:38 
semilla 700057  fin 22:58:50 


In [57]:
# --- validation: cada semilla y el ensamble ---------------------------------------
# ensamble en validation = promedio de las probabilidades de las semillas
prob_val_ens <- rowMeans( do.call( cbind, lista_val ) )

lista_tb[[ "ensamble" ]] <- data.table(
  semilla= "ensamble",
  auc_val= AUC_rank( prob_val_ens, dval$clase01 ),
  envios= tb_cortes_val$envios,
  ganancia_val= ganancia_cortes( prob_val_ens, dval$clase_ternaria, tb_cortes_val$envios_val )
)

tb_val <- rbindlist( lista_tb )
fwrite( tb_val, file= "tb_semillas_validacion.txt", sep= "\t" )

# resumen: AUC y ganancia media sobre los cortes fijos, por semilla y para el ensamble
resumen_val <- tb_val[, list(
  auc_val= auc_val[1],
  ganancia_media_cortes= mean(ganancia_val),
  ganancia_max= max(ganancia_val) ),
  by= semilla ]

fwrite( resumen_val, file= "tb_resumen_validacion.txt", sep= "\t" )
resumen_val

semilla,auc_val,ganancia_media_cortes,ganancia_max
<chr>,<dbl>,<dbl>,<dbl>
165463,0.9344030,95179545,98775000
329969,0.9343080,96997727,99725000
454253,0.9323582,94815909,96225000
593899,0.9326016,95543182,97275000
980249,0.9316807,94634091,99200000
100183,0.9332617,94543182,97275000
250027,0.9322398,95634091,96750000
400031,0.9322741,95270455,98175000
550049,0.9326691,97725000,100225000


#### Scoring y ensamble

Cada semilla ya aplico su modelo final a 202109 (dentro del bucle) y guardo `prediccion_<semilla>.txt`.
Aqui se **promedian las probabilidades** de las semillas: el ensamble reduce el ruido de una sola semilla.

##### Tabla Prediccion

In [58]:
# ensamble de las semillas: promedio de las probabilidades
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := rowMeans( do.call( cbind, lista_prob ) ) ]

# grabo las probabilidades del ensamble
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

cat( "semillas en el ensamble:", length(lista_prob), "\n" )

semillas en el ensamble: 10 


#### Kaggle Competition Submit

Se envia el **ensamble** en los **cortes fijos** `PARAM$kaggle$cortes` (1600 a 2600 cada 100 = 11 envios por metodo).
Los cortes son iguales para todos los metodos, asi que los puntajes de Kaggle son comparables.
Kaggle acepta 100 envios por dia y se resetea a las 21 hs de Argentina.

In [59]:
# genero los archivos con los "envios" y los subo a Kaggle
#  (PARAM$kaggle$enviar = FALSE genera los archivos sin subirlos)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marco los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  if( PARAM$kaggle$enviar ) {

    # subida a Kaggle, armo la linea de comando
    comando <- "kaggle competitions submit"
    competencia <- paste("-c", PARAM$kaggle$competencia)
    arch <- paste( "-f", archivo_kaggle)

    mensaje <- paste0("-m 'DR=", PARAM$DR$metodo,
      "  ensamble=", length(lista_prob), "sem",
      "  envios=", envios,
      "'" )

    linea <- paste( comando, competencia, arch, mensaje)

    # si un envio falla, se informa y se sigue con el siguiente
    salida <- tryCatch(
      system(linea, intern=TRUE), # el submit a Kaggle
      error= function(e) conditionMessage(e),
      warning= function(w) conditionMessage(w)
    )
    Sys.sleep(30)
    cat(envios, ":", salida, "\n")
  }
}

1600 : Warning: Looks like you're using an outdated `kaggle` version (installed: 2.2.4), please consider upgrading to the latest version (2.2.5) 99 submissions remaining today. Successfully submitted to Labo 1 Virtual 2026 analista Jr 
1700 : Warning: Looks like you're using an outdated `kaggle` version (installed: 2.2.4), please consider upgrading to the latest version (2.2.5) 98 submissions remaining today. Successfully submitted to Labo 1 Virtual 2026 analista Jr 
1800 : Warning: Looks like you're using an outdated `kaggle` version (installed: 2.2.4), please consider upgrading to the latest version (2.2.5) 97 submissions remaining today. Successfully submitted to Labo 1 Virtual 2026 analista Jr 
1900 : Warning: Looks like you're using an outdated `kaggle` version (installed: 2.2.4), please consider upgrading to the latest version (2.2.5) 96 submissions remaining today. Successfully submitted to Labo 1 Virtual 2026 analista Jr 
2000 : Warning: Looks like you're using an outdated `kag

In [60]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [61]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Fri Nov 06 23:04:47 2026"